# Retail Sales & Customer Insights — Full Pipeline
**Phases covered in this notebook:**
1. MySQL connection setup & staging table creation
2. Load raw data into staging tables
3. Star-schema DDL (Dimension + Fact tables)
4. ETL — populate Dimension then Fact tables
5. Python data cleaning & feature engineering
6. Export Power BI–ready CSVs (`powerbi_data/`)
7. KPI summary preview

> **Before running:** Update the MySQL credentials in Cell 2.

## Cell 1 — Install dependencies (run once)

In [1]:
# Run this cell once to install required packages
# If already installed, it will just confirm quickly
import subprocess, sys
for pkg in ["sqlalchemy", "pymysql"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])
print("✓ Dependencies ready")

✓ Dependencies ready


## Cell 2 — MySQL Connection Configuration
> **Edit the values below to match your MySQL server.**

In [2]:
# ── MySQL credentials — EDIT THESE ──────────────────────────
DB_HOST     = "127.0.0.1"
DB_PORT     = 3306
DB_USER     = "root"
DB_PASSWORD = "Monika#80"   # ← change this
DB_NAME     = "retail_dw"       # ← database must already exist in MySQL
# ─────────────────────────────────────────────────────────────

# File paths — update if your files are in a different folder
SALES_CSV     = "sales 1.csv"
CUSTOMERS_JSON= "customers.json"
PRODUCTS_CSV  = "products.csv"

print(f"Target DB: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

Target DB: root@127.0.0.1:3306/retail_dw


In [3]:
from sqlalchemy import create_engine, text

DB_NAME = "retail_dw"

# Connect to MySQL without specifying database
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/",
    echo=False
)

# Create database if it doesn't exist
with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✓ Database {DB_NAME} is ready")

# Now connect to the newly created database
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("✓ Connected to MySQL successfully")

✓ Database retail_dw is ready
✓ Connected to MySQL successfully


## Cell 3 — Imports & DB Engine

In [4]:
import pandas as pd
import os
from sqlalchemy import create_engine, text

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)

# Quick connectivity test
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("✓ Connected to MySQL successfully")

✓ Connected to MySQL successfully


## Cell 4 — Load Raw Data Files into Python

In [5]:
# ── Load raw files ───────────────────────────────────────────
sales     = pd.read_csv(SALES_CSV)
customers = pd.read_json(CUSTOMERS_JSON)
products  = pd.read_csv(PRODUCTS_CSV)

print("=== SALES ===")
print(f"Shape: {sales.shape}  |  Columns: {list(sales.columns)}")
print(sales.head(3))

print("\n=== CUSTOMERS ===")
print(f"Shape: {customers.shape}  |  Columns: {list(customers.columns)}")
print(customers.head(3))

print("\n=== PRODUCTS ===")
print(f"Shape: {products.shape}  |  Columns: {list(products.columns)}")
print(products.head(3))

=== SALES ===
Shape: (10000, 6)  |  Columns: ['SaleID', 'ProductID', 'CustomerID', 'SalesAmount', 'Quantity', 'Timestamp']
                                 SaleID  ProductID CustomerID  SalesAmount  \
0  c0078cd1-3ddc-439a-b41d-7498e8e75e62       1020      C0460      1639.10   
1  be8532af-16b7-432c-a13d-4c6e6836d12d       1029      C0198       186.11   
2  8666f277-cc58-4fde-9d58-697da9e040f2       1012      C0524      1712.72   

   Quantity            Timestamp  
0      18.0  2024-08-09T20:21:00  
1      42.0  2024-11-28T05:01:00  
2      49.0  2024-10-13T00:36:00  

=== CUSTOMERS ===
Shape: (1100, 6)  |  Columns: ['CustomerID', 'FirstName', 'LastName', 'Gender', 'Region', 'SSN']
  CustomerID FirstName LastName Gender      Region          SSN
0      C0001   Gregory   Miller      M        Ohho    082573197
1      C0002    Marvin     None   male  california  309-6256-21
2      C0003   Gregory    Smith   Male   New Yorkk    407071725

=== PRODUCTS ===
Shape: (50, 3)  |  Columns: ['Prod

## Cell 5 — Data Cleaning (Phase 2 / Phase 3 Python step)

In [ ]:
# ── Standardize column names ─────────────────────────────────
sales.columns     = sales.columns.str.lower().str.replace(' ', '_')
customers.columns = customers.columns.str.lower().str.replace(' ', '_')
products.columns  = products.columns.str.lower().str.replace(' ', '_')
# After rename:
#   sales     → saleid, productid, customerid, salesamount, quantity, timestamp
#   customers → customerid, firstname, lastname, gender, region, ssn
#   products  → productid, productname, category

# ── Remove duplicates ─────────────────────────────────────────
for df in [sales, customers, products]:
    df.drop_duplicates(inplace=True)

# ── Handle missing values ─────────────────────────────────────
for df in [sales, customers, products]:
    for col in df.select_dtypes(include='number').columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(include='object').columns:
        if not df[col].mode().empty:
            df[col] = df[col].fillna(df[col].mode().iloc[0])

# ── Parse timestamp ──────────────────────────────────────────
sales['timestamp'] = pd.to_datetime(sales['timestamp'], errors='coerce')
sales.dropna(subset=['salesamount'], inplace=True)

print("Null counts after cleaning:")
print("Sales:    ", sales.isnull().sum().sum())
print("Customers:", customers.isnull().sum().sum())
print("Products: ", products.isnull().sum().sum())
print("✓ Cleaning complete")

Null counts after cleaning:
Sales:     0
Customers: 0
Products:  0
✓ Cleaning complete


## Cell 6 — Feature Engineering

In [ ]:
# ── Temporal features ────────────────────────────────────────
sales['order_year']  = sales['timestamp'].dt.year
sales['order_month'] = sales['timestamp'].dt.month
sales['order_date']  = sales['timestamp'].dt.date
# ── Unit price (back-calculated; no price column in source) ──
sales['unit_price'] = (sales['salesamount'] / sales['quantity']).round(2)
# ── Customer derived fields ───────────────────────────────────
# Handle null lastnames: use firstname only if lastname is null/NaN
customers['customer_name'] = customers.apply(
    lambda row: row['firstname'] if pd.isna(row['lastname']) 
                else f"{row['firstname']} {row['lastname']}", 
    axis=1
)
# Normalize gender to handle all variations (M/Male/male → Male, F/Female/female → Female)
customers['gender_normalized'] = customers['gender'].str.upper().str[0]  
customers['segment'] = customers['gender_normalized'].map(
    {'M': 'Male', 'F': 'Female'}
).fillna('Other')
# ── Product price_range (derived from avg sales unit_price) ──
avg_price = (
    sales.groupby('productid')['unit_price']
    .mean().reset_index().rename(columns={'unit_price': 'avg_price'})
)
products = products.merge(avg_price, on='productid', how='left')
products['avg_price'].fillna(products['avg_price'].median(), inplace=True)
products['price_range'] = pd.cut(
    products['avg_price'],
    bins=[0, 50, 100, 200, float('inf')],
    labels=['Budget', 'Standard', 'Premium', 'Luxury']
)

print("✓ Feature engineering complete")
print(f"Sales shape: {sales.shape} | new cols: order_year, order_month, order_date, unit_price")
print(f"Customers shape: {customers.shape} | new cols: customer_name, segment, age_group")
print(f"Products shape: {products.shape} | new cols: avg_price, price_range")

✓ Feature engineering complete
Sales shape: (10000, 10) | new cols: order_year, order_month, order_date, unit_price
Customers shape: (1000, 10) | new cols: customer_name, segment, age_group
Products shape: (50, 5) | new cols: avg_price, price_range


C:\Users\harsh\AppData\Local\Temp\ipykernel_20344\2643002378.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  products['avg_price'].fillna(products['avg_price'].median(), inplace=True)


## Cell 7 — Create Staging Tables in MySQL
Staging tables are the "landing zone" for raw transformed data.  
The ETL SQL in Cell 9 reads *from* these tables.  
`if_exists='replace'` means re-running is safe — the table is dropped and recreated.

In [8]:
# Write cleaned DataFrames directly into MySQL as staging tables
staging_sales     = sales.copy()
staging_customers = customers.copy()
staging_products  = products.copy()

# price_range is a Categorical — convert to string before writing to SQL
staging_products['price_range'] = staging_products['price_range'].astype(str)

staging_sales.to_sql(    'staging_sales',     engine, if_exists='replace', index=False)
staging_customers.to_sql('staging_customers', engine, if_exists='replace', index=False)
staging_products.to_sql( 'staging_products',  engine, if_exists='replace', index=False)

print("✓ staging_sales     →", len(staging_sales),     "rows")
print("✓ staging_customers →", len(staging_customers), "rows")
print("✓ staging_products  →", len(staging_products),  "rows")

✓ staging_sales     → 10000 rows
✓ staging_customers → 1000 rows
✓ staging_products  → 50 rows


## Cell 8 — Star Schema DDL
Creates Dimension and Fact tables.  
Safe to re-run — uses `DROP TABLE IF EXISTS` before each `CREATE`.

In [9]:
ddl_statements = [

    # ── Drop existing tables (Fact first, then Dimensions) ────────────────
    "DROP TABLE IF EXISTS FACT_SALES",
    "DROP TABLE IF EXISTS DIM_DATE",
    "DROP TABLE IF EXISTS DIM_PRODUCT",
    "DROP TABLE IF EXISTS DIM_CUSTOMER",

    # ── DIM_CUSTOMER ──────────────────────────────────────────────────────
    """CREATE TABLE DIM_CUSTOMER (
        customer_id   VARCHAR(20)  PRIMARY KEY,
        customer_name VARCHAR(255) NOT NULL,
        firstname     VARCHAR(50),
        lastname      VARCHAR(50),
        gender        VARCHAR(10),
        segment       VARCHAR(20),
        region        VARCHAR(100),
        age_group     VARCHAR(100)
    )""",
    "CREATE INDEX idx_customer_region  ON DIM_CUSTOMER(region)",
    "CREATE INDEX idx_customer_segment ON DIM_CUSTOMER(segment)",

    # ── DIM_PRODUCT ───────────────────────────────────────────────────────
    """CREATE TABLE DIM_PRODUCT (
        product_id   INT          PRIMARY KEY,
        product_name VARCHAR(255) NOT NULL,
        category     VARCHAR(100),
        avg_price    DECIMAL(10,2),
        price_range  VARCHAR(20)
    )""",
    "CREATE INDEX idx_product_category ON DIM_PRODUCT(category)",

    # ── DIM_DATE ──────────────────────────────────────────────────────────
    """CREATE TABLE DIM_DATE (
        date_id      INT  PRIMARY KEY,
        full_date    DATE UNIQUE NOT NULL,
        year         INT,
        month_number INT,
        month_name   VARCHAR(10),
        quarter      INT
    )""",
    "CREATE INDEX idx_date_year_month ON DIM_DATE(year, month_number)",

    # ── FACT_SALES ────────────────────────────────────────────────────────
    """CREATE TABLE FACT_SALES (
        sale_id      VARCHAR(50)   PRIMARY KEY,
        customer_id  VARCHAR(20)   NOT NULL,
        product_id   INT           NOT NULL,
        date_id      INT           NOT NULL,
        quantity     INT,
        sales_amount DECIMAL(12,2),
        unit_price   DECIMAL(10,2),
        FOREIGN KEY (customer_id) REFERENCES DIM_CUSTOMER(customer_id),
        FOREIGN KEY (product_id)  REFERENCES DIM_PRODUCT(product_id),
        FOREIGN KEY (date_id)     REFERENCES DIM_DATE(date_id)
    )""",
    "CREATE INDEX idx_fact_customer ON FACT_SALES(customer_id)",
    "CREATE INDEX idx_fact_product  ON FACT_SALES(product_id)",
    "CREATE INDEX idx_fact_date     ON FACT_SALES(date_id)",
]

with engine.connect() as conn:
    for stmt in ddl_statements:
        conn.execute(text(stmt))
    conn.commit()

print("✓ Star schema created: DIM_CUSTOMER, DIM_PRODUCT, DIM_DATE, FACT_SALES")

✓ Star schema created: DIM_CUSTOMER, DIM_PRODUCT, DIM_DATE, FACT_SALES


## Cell 9 — ETL: Populate Dimension Tables
Reads from staging tables and inserts into the star-schema Dimensions.  
Dimension tables must be loaded **before** FACT_SALES (foreign key constraint).

In [10]:
etl_dim_sql = [

    # ── DIM_CUSTOMER ──────────────────────────────────────────────────────
    """INSERT INTO DIM_CUSTOMER
        (customer_id, customer_name, firstname, lastname, gender, segment, region, age_group)
    SELECT
        customerid,
        customer_name,
        firstname,
        lastname,
        segment,          -- Male/Female/Other (normalized in Python, not raw M/F)
        segment,
        region,
        age_group
    FROM staging_customers""",

    # ── DIM_PRODUCT ───────────────────────────────────────────────────────
    """INSERT INTO DIM_PRODUCT
        (product_id, product_name, category, avg_price, price_range)
    SELECT
        p.productid,
        p.productname,
        p.category,
        ROUND(p.avg_price, 2),
        p.price_range
    FROM staging_products p""",

    # ── DIM_DATE ──────────────────────────────────────────────────────────
    """INSERT INTO DIM_DATE
        (date_id, full_date, year, month_number, month_name, quarter)
    SELECT DISTINCT
        CAST(DATE_FORMAT(DATE(timestamp), '%Y%m%d') AS UNSIGNED),
        DATE(timestamp),
        YEAR(timestamp),
        MONTH(timestamp),
        MONTHNAME(timestamp),
        QUARTER(timestamp)
    FROM staging_sales
    WHERE timestamp IS NOT NULL""",
]

with engine.connect() as conn:
    for stmt in etl_dim_sql:
        conn.execute(text(stmt))
    conn.commit()

print("✓ DIM_CUSTOMER, DIM_PRODUCT, DIM_DATE populated")

✓ DIM_CUSTOMER, DIM_PRODUCT, DIM_DATE populated


## Cell 10 — ETL: Populate FACT_SALES

In [ ]:
# ── ETL FACT_SALES ── Join staging_sales with dimension tables to get surrogate keys and insert into FACT_SALES
etl_fact_sql = """
INSERT INTO FACT_SALES
    (sale_id, customer_id, product_id, date_id, quantity, sales_amount, unit_price)
SELECT
    s.saleid,
    c.customer_id,
    p.product_id,
    d.date_id,
    s.quantity,
    s.salesamount,
    ROUND(s.salesamount / NULLIF(s.quantity, 0), 2)
FROM staging_sales s
INNER JOIN DIM_CUSTOMER c
    ON s.customerid = c.customer_id
INNER JOIN DIM_PRODUCT p
    ON s.productid = p.product_id
INNER JOIN DIM_DATE d
    ON CAST(DATE_FORMAT(DATE(s.timestamp), '%Y%m%d') AS UNSIGNED) = d.date_id
WHERE s.salesamount IS NOT NULL
  AND s.quantity > 0
"""

with engine.connect() as conn:
    conn.execute(text(etl_fact_sql))
    conn.commit()

# Verify row count
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM FACT_SALES")).scalar()
print(f"✓ FACT_SALES populated: {count:,} rows")

✓ FACT_SALES populated: 10,000 rows


## Cell 11 — Create Reporting Views

In [12]:
views_sql = [

    ("DROP VIEW IF EXISTS vw_sales_overview", None),
    ("DROP VIEW IF EXISTS vw_customer_insights", None),
    ("DROP VIEW IF EXISTS vw_product_performance", None),
    ("DROP VIEW IF EXISTS vw_monthly_sales_trend", None),
    ("DROP VIEW IF EXISTS vw_regional_sales", None),

    ("vw_sales_overview", """
CREATE VIEW vw_sales_overview AS
SELECT
    d.full_date, d.year, d.month_number, d.month_name, d.quarter,
    c.customer_id, c.customer_name, c.region, c.segment, c.age_group,
    p.product_id, p.product_name, p.category, p.price_range,
    s.sale_id, s.quantity, s.unit_price, s.sales_amount
FROM FACT_SALES s
INNER JOIN DIM_DATE     d ON s.date_id     = d.date_id
INNER JOIN DIM_CUSTOMER c ON s.customer_id = c.customer_id
INNER JOIN DIM_PRODUCT  p ON s.product_id  = p.product_id"""),

    ("vw_customer_insights", """
CREATE VIEW vw_customer_insights AS
SELECT
    c.customer_id, c.customer_name, c.region, c.segment, c.age_group,
    COUNT(s.sale_id)    AS total_orders,
    SUM(s.sales_amount) AS total_spent,
    AVG(s.sales_amount) AS avg_order_value
FROM FACT_SALES s
INNER JOIN DIM_CUSTOMER c ON s.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name, c.region, c.segment, c.age_group"""),

    ("vw_product_performance", """
CREATE VIEW vw_product_performance AS
SELECT
    p.product_id, p.product_name, p.category, p.price_range,
    SUM(s.quantity)     AS total_quantity,
    SUM(s.sales_amount) AS total_sales,
    RANK() OVER (ORDER BY SUM(s.sales_amount) DESC) AS sales_rank
FROM FACT_SALES s
INNER JOIN DIM_PRODUCT p ON s.product_id = p.product_id
GROUP BY p.product_id, p.product_name, p.category, p.price_range"""),

    ("vw_monthly_sales_trend", """
CREATE VIEW vw_monthly_sales_trend AS
SELECT
    d.year, d.month_number, d.month_name,
    SUM(s.sales_amount) AS monthly_sales,
    LAG(SUM(s.sales_amount)) OVER (ORDER BY d.year, d.month_number) AS previous_month_sales,
    ROUND(
        (SUM(s.sales_amount) - LAG(SUM(s.sales_amount)) OVER (ORDER BY d.year, d.month_number))
        / NULLIF(LAG(SUM(s.sales_amount)) OVER (ORDER BY d.year, d.month_number), 0) * 100
    , 2) AS mom_growth_pct
FROM FACT_SALES s
INNER JOIN DIM_DATE d ON s.date_id = d.date_id
GROUP BY d.year, d.month_number, d.month_name"""),

    ("vw_regional_sales", """
CREATE VIEW vw_regional_sales AS
SELECT
    c.region,
    COUNT(s.sale_id)            AS total_orders,
    SUM(s.sales_amount)         AS total_sales,
    AVG(s.sales_amount)         AS avg_order_value,
    COUNT(DISTINCT s.customer_id) AS unique_customers
FROM FACT_SALES s
INNER JOIN DIM_CUSTOMER c ON s.customer_id = c.customer_id
GROUP BY c.region"""),
]

with engine.connect() as conn:
    for item in views_sql:
        if item[1] is None:                  # DROP statement
            conn.execute(text(item[0]))
        else:                                # CREATE VIEW
            conn.execute(text(item[1]))
    conn.commit()

print("✓ Views created: vw_sales_overview, vw_customer_insights,")
print("                 vw_product_performance, vw_monthly_sales_trend, vw_regional_sales")

✓ Views created: vw_sales_overview, vw_customer_insights,
                 vw_product_performance, vw_monthly_sales_trend, vw_regional_sales


## Phase 4 — Power BI Data Layer
Create MySQL Views and KPI Summary Table for direct Power BI connection (no CSV files).

In [13]:
# ══════════════════════════════════════════════════════════════════
#  Phase 4 — Power BI Data Layer (MySQL Views + KPI Table)
#  All CSV exports removed. Power BI connects directly to MySQL.
# ══════════════════════════════════════════════════════════════════

from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)

# ── Drop & recreate all Power BI views + KPI summary table ────────
powerbi_ddl = [

    # ── Drop existing objects (views first, then table) ───────────
    "DROP VIEW IF EXISTS vw_sales_overview",
    "DROP VIEW IF EXISTS vw_customer_insights",
    "DROP VIEW IF EXISTS vw_product_performance",
    "DROP VIEW IF EXISTS vw_monthly_sales_trend",
    "DROP VIEW IF EXISTS vw_regional_sales",
    "DROP VIEW IF EXISTS vw_category_sales",
    "DROP VIEW IF EXISTS vw_yearly_sales_growth",
    "DROP TABLE IF EXISTS vw_kpi_summary",

    # ── vw_sales_overview ─────────────────────────────────────────
    # Flat denormalised view: one row per transaction with all
    # dimension attributes. Used as the main canvas in Power BI.
    """CREATE VIEW vw_sales_overview AS
SELECT
    d.full_date,
    d.year                          AS order_year,
    d.month_number                  AS order_month,
    d.month_name,
    d.quarter,
    c.customer_id,
    c.customer_name,
    c.region,
    c.segment,
    c.age_group,
    p.product_id,
    p.product_name,
    p.category,
    p.price_range,
    f.sale_id,
    f.quantity,
    f.unit_price,
    f.sales_amount
FROM FACT_SALES f
INNER JOIN DIM_DATE     d ON f.date_id     = d.date_id
INNER JOIN DIM_CUSTOMER c ON f.customer_id = c.customer_id
INNER JOIN DIM_PRODUCT  p ON f.product_id  = p.product_id""",

    # ── vw_customer_insights ──────────────────────────────────────
    # KPI: Customer Lifetime Value (CLV) — aggregated per customer.
    # Feeds the Customer Demographics and CLV visuals in Power BI.
    """CREATE VIEW vw_customer_insights AS
SELECT
    c.customer_id,
    c.customer_name,
    c.region,
    c.segment,
    c.age_group,
    COUNT(f.sale_id)                        AS total_orders,
    ROUND(SUM(f.sales_amount), 2)           AS total_spent,
    ROUND(AVG(f.sales_amount), 2)           AS avg_order_value
FROM FACT_SALES f
INNER JOIN DIM_CUSTOMER c ON f.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name, c.region, c.segment, c.age_group""",

    # ── vw_product_performance ────────────────────────────────────
    # KPI: Top-Selling Products — ranked by revenue.
    # Feeds bar charts and Top-N product tables in Power BI.
    """CREATE VIEW vw_product_performance AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.price_range,
    SUM(f.quantity)                                          AS total_quantity,
    ROUND(SUM(f.sales_amount), 2)                           AS total_sales,
    RANK() OVER (ORDER BY SUM(f.sales_amount) DESC)         AS sales_rank
FROM FACT_SALES f
INNER JOIN DIM_PRODUCT p ON f.product_id = p.product_id
GROUP BY p.product_id, p.product_name, p.category, p.price_range""",

    # ── vw_monthly_sales_trend ────────────────────────────────────
    # KPI: Sales Growth Rate (Monthly MoM %).
    # Feeds the line chart showing monthly/seasonal sales trend.
    """CREATE VIEW vw_monthly_sales_trend AS
SELECT
    d.year,
    d.month_number,
    d.month_name,
    ROUND(SUM(f.sales_amount), 2)                           AS monthly_sales,
    ROUND(LAG(SUM(f.sales_amount))
          OVER (ORDER BY d.year, d.month_number), 2)        AS previous_month_sales,
    ROUND(
        (SUM(f.sales_amount)
         - LAG(SUM(f.sales_amount)) OVER (ORDER BY d.year, d.month_number))
        / NULLIF(LAG(SUM(f.sales_amount))
                 OVER (ORDER BY d.year, d.month_number), 0) * 100
    , 2)                                                    AS mom_growth_pct
FROM FACT_SALES f
INNER JOIN DIM_DATE d ON f.date_id = d.date_id
GROUP BY d.year, d.month_number, d.month_name""",

    # ── vw_regional_sales ─────────────────────────────────────────
    # KPI: Regional Sales Performance.
    # Feeds the map visual and regional bar chart in Power BI.
    """CREATE VIEW vw_regional_sales AS
SELECT
    c.region,
    COUNT(f.sale_id)                        AS total_orders,
    ROUND(SUM(f.sales_amount), 2)           AS total_sales,
    ROUND(AVG(f.sales_amount), 2)           AS avg_order_value,
    COUNT(DISTINCT f.customer_id)           AS unique_customers
FROM FACT_SALES f
INNER JOIN DIM_CUSTOMER c ON f.customer_id = c.customer_id
GROUP BY c.region""",

    # ── vw_category_sales ─────────────────────────────────────────
    # KPI: Sales by Product Category.
    # Feeds the pie / donut chart in Power BI.
    """CREATE VIEW vw_category_sales AS
SELECT
    p.category,
    ROUND(SUM(f.sales_amount), 2)                           AS total_sales,
    SUM(f.quantity)                                         AS total_quantity,
    COUNT(f.sale_id)                                        AS total_orders,
    ROUND(SUM(f.sales_amount)
          / SUM(SUM(f.sales_amount)) OVER () * 100, 2)     AS sales_pct
FROM FACT_SALES f
INNER JOIN DIM_PRODUCT p ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC""",

    # ── vw_yearly_sales_growth ────────────────────────────────────
    # KPI: Sales Growth Rate (Year-on-Year %).
    # Feeds the YoY growth card / column chart in Power BI.
    """CREATE VIEW vw_yearly_sales_growth AS
SELECT
    d.year                                                  AS order_year,
    ROUND(SUM(f.sales_amount), 2)                           AS yearly_sales,
    ROUND(LAG(SUM(f.sales_amount)) OVER (ORDER BY d.year),
          2)                                                AS prev_year_sales,
    ROUND(
        (SUM(f.sales_amount)
         - LAG(SUM(f.sales_amount)) OVER (ORDER BY d.year))
        / NULLIF(LAG(SUM(f.sales_amount)) OVER (ORDER BY d.year), 0) * 100
    , 2)                                                    AS yoy_growth_pct
FROM FACT_SALES f
INNER JOIN DIM_DATE d ON f.date_id = d.date_id
GROUP BY d.year""",
]

with engine.connect() as conn:
    for stmt in powerbi_ddl:
        conn.execute(text(stmt))
    conn.commit()

print("✓ Power BI views created in MySQL:")
views = [
    "vw_sales_overview", "vw_customer_insights", "vw_product_performance",
    "vw_monthly_sales_trend", "vw_regional_sales", "vw_category_sales",
    "vw_yearly_sales_growth",
]
for v in views:
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {v}")).scalar()
    print(f"  {v:<35} {n:>6} rows")

# ── vw_kpi_summary — stored as a table so Power BI can use it ─────
# (MySQL does not support scalar subqueries well inside a single-row view)
with engine.connect() as conn:
    total_rev   = conn.execute(text("SELECT ROUND(SUM(sales_amount),2) FROM FACT_SALES")).scalar()
    n_orders    = conn.execute(text("SELECT COUNT(*) FROM FACT_SALES")).scalar()
    n_cust      = conn.execute(text("SELECT COUNT(DISTINCT customer_id) FROM FACT_SALES")).scalar()
    latest_yoy  = conn.execute(text(
        "SELECT yoy_growth_pct FROM vw_yearly_sales_growth "
        "WHERE yoy_growth_pct IS NOT NULL ORDER BY order_year DESC LIMIT 1"
    )).scalar()

kpi_df = pd.DataFrame([{
    "total_revenue":           float(total_rev),
    "total_orders":            int(n_orders),
    "avg_transaction_value":   round(float(total_rev) / int(n_orders), 2),
    "unique_customers":        int(n_cust),
    "customer_lifetime_value": round(float(total_rev) / int(n_cust), 2),
    "sales_growth_rate_pct":   float(latest_yoy) if latest_yoy is not None else None,
}])

kpi_df.to_sql("vw_kpi_summary", engine, if_exists="replace", index=False)
print(f"  {'vw_kpi_summary (table)':<35}      1 row")

print()
print("══════════════════════════════════════════════════════════════")
print("  Power BI Connection Instructions")
print("══════════════════════════════════════════════════════════════")
print(f"  Connector : MySQL Database")
print(f"  Server    : {DB_HOST}:{DB_PORT}")
print(f"  Database  : {DB_NAME}")
print(f"  Mode      : Import  (or DirectQuery for live refresh)")
print()
print("  Tables / Views to import:")
for v in views + ["vw_kpi_summary"]:
    print(f"    • {v}")
print()
print("  In Power BI Desktop:")
print("  Home → Get Data → MySQL Database → enter server & DB")
print("  Select the views above → Load (Import) or Transform Data")
print("══════════════════════════════════════════════════════════════")


✓ Power BI views created in MySQL:
  vw_sales_overview                    10000 rows
  vw_customer_insights                  1000 rows
  vw_product_performance                  50 rows
  vw_monthly_sales_trend                   6 rows
  vw_regional_sales                       10 rows
  vw_category_sales                        4 rows
  vw_yearly_sales_growth                   1 rows
  vw_kpi_summary (table)                   1 row

══════════════════════════════════════════════════════════════
  Power BI Connection Instructions
══════════════════════════════════════════════════════════════
  Connector : MySQL Database
  Server    : 127.0.0.1:3306
  Database  : retail_dw
  Mode      : Import  (or DirectQuery for live refresh)

  Tables / Views to import:
    • vw_sales_overview
    • vw_customer_insights
    • vw_product_performance
    • vw_monthly_sales_trend
    • vw_regional_sales
    • vw_category_sales
    • vw_yearly_sales_growth
    • vw_kpi_summary

  In Power BI Desktop:
  Home

## Verification — Query MySQL Views

In [14]:
# ══════════════════════════════════════════════════════════════════
#  Verification: Query MySQL views to confirm data loaded correctly
# ══════════════════════════════════════════════════════════════════

from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)

# ── KPI Summary ───────────────────────────────────────────────────
kpi = pd.read_sql("SELECT * FROM vw_kpi_summary", engine).iloc[0]

print("=" * 55)
print("  KPI SUMMARY  (sourced from MySQL · vw_kpi_summary)")
print("=" * 55)
print(f"  Total Revenue            : ₹{kpi['total_revenue']:>15,.2f}")
print(f"  Total Orders             : {int(kpi['total_orders']):>15,}")
print(f"  Avg Transaction Value    : ₹{kpi['avg_transaction_value']:>15,.2f}")
print(f"  Unique Customers         : {int(kpi['unique_customers']):>15,}")
print(f"  Customer Lifetime Value  : ₹{kpi['customer_lifetime_value']:>15,.2f}")
yoy = kpi.get("sales_growth_rate_pct")
yoy_str = f"{yoy:>14.2f}%" if pd.notna(yoy) else "N/A"
print(f"  YoY Sales Growth Rate    : {yoy_str}")
print()

print("Top 5 Products by Revenue  (vw_product_performance):")
display(pd.read_sql(
    "SELECT sales_rank, product_name, category, total_sales "
    "FROM vw_product_performance ORDER BY sales_rank LIMIT 5", engine
))

print("\nMonthly Sales Trend  (vw_monthly_sales_trend):")
display(pd.read_sql(
    "SELECT year, month_number, month_name, monthly_sales, mom_growth_pct "
    "FROM vw_monthly_sales_trend ORDER BY year, month_number", engine
))

print("\nRegional Sales  (vw_regional_sales):")
display(pd.read_sql(
    "SELECT region, total_orders, total_sales, avg_order_value, unique_customers "
    "FROM vw_regional_sales ORDER BY total_sales DESC", engine
))

print("\nSales by Category  (vw_category_sales):")
display(pd.read_sql(
    "SELECT category, total_sales, total_quantity, sales_pct "
    "FROM vw_category_sales ORDER BY total_sales DESC", engine
))

print("\nYear-on-Year Growth  (vw_yearly_sales_growth):")
display(pd.read_sql(
    "SELECT order_year, yearly_sales, prev_year_sales, yoy_growth_pct "
    "FROM vw_yearly_sales_growth ORDER BY order_year", engine
))


  KPI SUMMARY  (sourced from MySQL · vw_kpi_summary)
  Total Revenue            : ₹  14,160,299.56
  Total Orders             :          10,000
  Avg Transaction Value    : ₹       1,416.03
  Unique Customers         :           1,000
  Customer Lifetime Value  : ₹      14,160.30
  YoY Sales Growth Rate    : N/A

Top 5 Products by Revenue  (vw_product_performance):


,sales_rank,product_name,category,total_sales
0,1,Action Figure,Toys,653328.88
1,2,Swing,Toys,413553.53
2,3,Stuffed Animal,Toys,407897.29
3,4,Board Game,Toys,403120.59
4,5,Doll,Toys,401508.61



Monthly Sales Trend  (vw_monthly_sales_trend):


,year,month_number,month_name,monthly_sales,mom_growth_pct
0,2024,6,June,2120907.19,NaN
1,2024,7,July,2289080.72,7.93
2,2024,8,August,2792319.57,21.98
3,2024,9,September,2182497.38,-21.84
4,2024,10,October,2424428.35,11.09
5,2024,11,November,2351066.35,-3.03



Regional Sales  (vw_regional_sales):


,region,total_orders,total_sales,avg_order_value,unique_customers
0,Californiya,1661,2303545.20,1386.84,164
1,New York,957,1640485.15,1714.20,96
2,Texas,898,1377572.29,1534.04,92
3,NY,1100,1374371.92,1249.43,102
4,Texaz,978,1373990.84,1404.90,102
5,New Yorkk,897,1342065.78,1496.17,87
6,Ohho,931,1320200.91,1418.05,94
7,Ohio,854,1190744.62,1394.31,81
8,california,879,1154170.63,1313.05,90
9,Nw York,845,1083152.22,1281.84,92



Sales by Category  (vw_category_sales):


,category,total_sales,total_quantity,sales_pct
0,Toys,6107951.46,103051.0,43.13
1,Electronics,5237296.29,97159.0,36.99
2,Furniture,1538247.57,26325.0,10.86
3,Home Appliances,1276804.24,25755.0,9.02



Year-on-Year Growth  (vw_yearly_sales_growth):


,order_year,yearly_sales,prev_year_sales,yoy_growth_pct
0,2024,14160299.56,None,None


## Cell 14 — Power BI Import Guide

### Option A — Import from CSV files (easiest, no MySQL needed)
1. Open **Power BI Desktop**
2. Click **Get Data → Text/CSV** and load each file from `powerbi_data/`:
   - `vw_sales_overview.csv`
   - `vw_customer_insights.csv`
   - `vw_product_performance.csv`
   - `vw_monthly_sales_trend.csv`
   - `vw_yearly_sales_growth.csv`
   - `vw_regional_sales.csv`
   - `vw_category_sales.csv`
   - `vw_kpi_summary.csv`

### Option B — Connect directly to MySQL
1. **Get Data → MySQL database**
2. Server: `127.0.0.1`, Database: `retail_dw`
3. Import: `FACT_SALES`, `DIM_CUSTOMER`, `DIM_PRODUCT`, `DIM_DATE` (or the views)

---

### Relationships (Model View)
After loading, go to **Model view** (left sidebar icon) and define:

| From table | From column | To table | To column | Cardinality |
|---|---|---|---|---|
| `vw_sales_overview` | `customerid` | `vw_customer_insights` | `customerid` | Many→One |
| `vw_sales_overview` | `productid` | `vw_product_performance` | `productid` | Many→One |
| `vw_sales_overview` | `order_year` | `vw_yearly_sales_growth` | `order_year` | Many→One |

---

### DAX Measures to create (Home → New Measure)
```dax
Total Revenue         = SUM(vw_sales_overview[salesamount])
Total Orders          = COUNTROWS(vw_sales_overview)
Avg Transaction Value = DIVIDE([Total Revenue], [Total Orders])
Unique Customers      = DISTINCTCOUNT(vw_sales_overview[customerid])
Customer LTV          = DIVIDE([Total Revenue], [Unique Customers])

Sales Growth Rate % =
    VAR cur  = CALCULATE([Total Revenue], YEAR(vw_sales_overview[timestamp]) = MAX(YEAR(vw_sales_overview[timestamp])))
    VAR prev = CALCULATE([Total Revenue], YEAR(vw_sales_overview[timestamp]) = MAX(YEAR(vw_sales_overview[timestamp])) - 1)
    RETURN DIVIDE(cur - prev, prev) * 100

MoM Growth % =
    VAR cur  = [Total Revenue]
    VAR prev = CALCULATE([Total Revenue],
                   DATEADD(vw_sales_overview[timestamp], -1, MONTH))
    RETURN DIVIDE(cur - prev, prev) * 100
```
